<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-10-tuning-and-evaluation/lesson-10.3-batch-routing/practice/GCP_Capstone_10.3_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 10.3 — Batch API + Model Routing

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup: SDK, Auth, and Vertex Client

Run this first. It installs the unified `google-genai` SDK, authenticates with Application Default Credentials (Colab), and builds the Vertex client every exercise below depends on. Set your own project id before running.

In [ ]:
!pip install -q google-genai google-cloud-storage google-cloud-bigquery

# Colab-only ADC auth (no API keys)
try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via Colab ADC')
except ImportError:
    print('Not on Colab - assuming ADC already configured (gcloud auth application-default login)')

from google import genai
from google.genai import types
from google.genai.types import CreateBatchJobConfig, GenerateContentConfig
import json, time, os

PROJECT  = 'documind-ai-YOUR-ID'   # <-- set your project id
LOCATION = 'us-central1'           # asia-south1 for India production
BUCKET   = 'documind-batch'
USD_INR  = 85                      # for any INR cost display

# Unified Gemini SDK on Vertex AI
client = genai.Client(enterprise=True, project=PROJECT, location=LOCATION)
print('SDK ready')

## Exercise 1: Build a batch JSONL

**Difficulty:** Easy

Create a 10-request JSONL with `id` + `request` fields. Validate parsing locally before upload.

1. Write a helper that turns a list of `{id, prompt}` docs into batch-format JSONL (each line = `id` + a `request` holding `contents` and `generation_config`).
2. Generate 10 sample DocuMind classification docs.
3. Re-read the file and `json.loads` every line to confirm it parses.

**Expected behaviour:** 10 valid JSON lines with `id` + `request` structure.

In [ ]:
# Cell 1 - Build a batch JSONL in Vertex batch-prediction format
def build_batch_jsonl(documents: list[dict], output_path: str):
    '''documents: [{'id': '...', 'prompt': '...'}, ...]'''
    with open(output_path, 'w') as f:
        for doc in documents:
            record = {
                'id': doc['id'],
                'request': {
                    'contents': [{
                        'role': 'user',
                        'parts': [{'text': doc['prompt']}]
                    }],
                    'generation_config': {
                        'temperature': 0.2,
                        'max_output_tokens': 200,
                    }
                }
            }
            f.write(json.dumps(record) + '\n')
    return output_path

# Sample DocuMind classification batch
docs = [
    {'id': f'doc-{i:03d}',
     'prompt': f'Classify as INVOICE/CONTRACT/REPORT: Sample doc {i} content...'}
    for i in range(10)
]
build_batch_jsonl(docs, 'classify-batch.jsonl')

# Validate parsing
with open('classify-batch.jsonl') as f:
    count = 0
    for line in f:
        json.loads(line)  # raises on syntax error
        count += 1
print(f'OK: {count} valid records in classify-batch.jsonl')

## Exercise 2: Submit batch from GCS

**Difficulty:** Easy

Use `client.batches.create()` with a GCS source and destination. Poll until the job reaches a terminal state.

1. Write `launch_batch_job()` that submits a JSONL in GCS to a GCS output directory via `CreateBatchJobConfig`.
2. Write `poll_until_complete()` that re-fetches the job with `client.batches.get()` until it hits a terminal state.
3. Show the example call (requires a real GCS bucket to execute).

**Expected behaviour:** Job completes, output JSONL appears in the GCS destination.

In [ ]:
# Cell 2 - Submit a batch job from GCS (template: needs real GCS URIs)
def launch_batch_job(model: str, gcs_input: str, gcs_output: str, display_name: str):
    '''Submit batch job from GCS JSONL to a GCS output directory.'''
    job = client.batches.create(
        model=model,
        src=gcs_input,
        config=CreateBatchJobConfig(
            dest=gcs_output,
            display_name=display_name
        ),
    )
    return job

def poll_until_complete(job, interval: int = 30):
    TERMINAL = {'JOB_STATE_SUCCEEDED', 'JOB_STATE_FAILED',
                'JOB_STATE_CANCELLED', 'JOB_STATE_PAUSED'}
    while str(job.state) not in TERMINAL:
        print(f'  State: {job.state}')
        time.sleep(interval)
        job = client.batches.get(name=job.name)
    print(f'Final: {job.state}')
    return job

print('Ready to submit batch jobs.')
print('Example call (requires real GCS URIs):')
print('  job = launch_batch_job("gemini-3.6-flash", "gs://bucket/in.jsonl", "gs://bucket/out/", "nightly")')
print('  job = poll_until_complete(job)')

## Exercise 3: Rule-based tier router

**Difficulty:** Easy

Write a function that maps `query` + `doc_length` to Flash-Lite / Flash / Pro.

1. Define keyword signal lists for simple vs. complex intent.
2. Route very long docs and complex reasoning to `gemini-3.1-pro-preview`, short simple lookups to `gemini-3.1-flash-lite`, everything else to `gemini-3.6-flash`.
3. Test on a spread of sample queries and print the chosen model.

**Expected behaviour:** Simple signals go to Lite, complex signals go to Pro, else Flash.

In [ ]:
# Cell 3 - Rule-based tier router (deterministic, zero latency overhead)
COMPLEX_SIGNALS = ['analyze', 'compare', 'evaluate', 'synthesize',
                   'why does', 'implications', 'reasoning']
SIMPLE_SIGNALS  = ['classify', 'extract', 'what is', 'list', 'translate',
                   'find', 'name', 'identify']

def rule_route(query: str, doc_length: int = 0) -> str:
    '''Map query + doc_length to the cheapest capable model.'''
    q = query.lower()
    has_simple  = any(s in q for s in SIMPLE_SIGNALS)
    has_complex = any(s in q for s in COMPLEX_SIGNALS)

    # Very long docs need Pro's larger effective context handling
    if doc_length > 50000:
        return 'gemini-3.1-pro-preview'
    # Complex reasoning signals
    if has_complex:
        return 'gemini-3.1-pro-preview'
    # Simple extraction/classification on short docs
    if has_simple and doc_length < 5000:
        return 'gemini-3.1-flash-lite'
    # Default: Flash handles everything else
    return 'gemini-3.6-flash'

tests = [
    ('Classify this invoice', 500),
    ('Summarize Q3 earnings', 8000),
    ('Analyze the strategic implications of the merger', 12000),
    ('Extract vendor name', 300),
    ('Compare compliance risks across these three contracts', 75000),
]
print(f'{"Query":<55} {"DocLen":>7}  Model')
print('-' * 85)
for q, dl in tests:
    print(f'{q:<55} {dl:>7}  {rule_route(q, dl)}')

## Exercise 4: LLM-based router

**Difficulty:** Medium

Use Flash-Lite as a complexity classifier. Log routing decisions for later analysis.

1. Send the query to `gemini-3.1-flash-lite` with a system instruction that forces a one-word SIMPLE / MEDIUM / COMPLEX label.
2. Map the label to a model (`flash-lite` / `flash` / `pro`).
3. Return `(tier, model)` and note the negligible routing cost.

**Expected behaviour:** Flash-Lite returns a SIMPLE/MEDIUM/COMPLEX label mapped to a model.

In [ ]:
# Cell 4 - LLM-based router: Flash-Lite as a cheap complexity classifier
def llm_route(query: str) -> tuple[str, str]:
    '''Returns (tier_label, model_name).'''
    response = client.models.generate_content(
        model='gemini-3.1-flash-lite',
        contents=query,
        config=GenerateContentConfig(
            system_instruction=(
                'Classify query complexity. Reply with ONE word only:\n'
                'SIMPLE  - factual lookup, classification, extraction, formatting\n'
                'MEDIUM  - summarization, Q&A, moderate analysis\n'
                'COMPLEX - multi-step reasoning, nuanced analysis, synthesis'
            ),
            temperature=0,
            max_output_tokens=10,
        ),
    )
    tier = response.text.strip().upper()
    model_map = {
        'SIMPLE':  'gemini-3.1-flash-lite',
        'MEDIUM':  'gemini-3.6-flash',
        'COMPLEX': 'gemini-3.1-pro-preview',
    }
    return tier, model_map.get(tier, 'gemini-3.6-flash')

# Requires a real project/ADC to execute the call:
# tier, model = llm_route('Summarize this contract')
# print(tier, '->', model)
print('LLM-based routing function ready')
print('~10 tokens routing cost = $0.000001 per query (negligible)')

## Exercise 5: Batch + routing pipeline

**Difficulty:** Medium

Classify 100 docs into tiers, partition into 3 JSONL files, then submit a batch job per tier.

1. Route every doc with `rule_route()` and bucket it into flash-lite / flash / pro.
2. Write one JSONL per non-empty tier with `build_batch_jsonl()`.
3. Return per-tier stats (count, file, model) ready to feed `launch_batch_job()`.

**Expected behaviour:** 3 batch jobs running in parallel, each on its tier's model.

In [ ]:
# Cell 6 - End-to-end: classify -> partition into per-tier JSONL -> submit per tier
def batch_with_routing(documents: list[dict]):
    '''documents: [{'id', 'prompt', 'doc_length'}, ...]'''
    tiers = {'flash-lite': [], 'flash': [], 'pro': []}

    # Step 1: classify each doc into a tier via the rule router
    for doc in documents:
        model = rule_route(doc['prompt'], doc.get('doc_length', 0))
        if 'lite' in model:  tiers['flash-lite'].append(doc)
        elif 'pro' in model: tiers['pro'].append(doc)
        else:                tiers['flash'].append(doc)

    # Step 2: write one JSONL per non-empty tier
    files = {}
    for tier, items in tiers.items():
        if not items:
            continue
        path = f'{tier}-batch.jsonl'
        build_batch_jsonl(items, path)
        files[tier] = path

    # Step 3: assemble per-tier submit stats (feed each into launch_batch_job)
    model_map = {
        'flash-lite': 'gemini-3.1-flash-lite',
        'flash':      'gemini-3.6-flash',
        'pro':        'gemini-3.1-pro-preview',
    }
    stats = {}
    for tier, path in files.items():
        stats[tier] = {'count': len(tiers[tier]), 'file': path, 'model': model_map[tier]}
    return stats

# Test with 100 sample docs
sample_docs = [
    {'id': f'd{i}', 'prompt': rq, 'doc_length': dl}
    for i, (rq, dl) in enumerate(
        [('Classify as invoice or receipt', 400) for _ in range(60)] +
        [('Summarize this Q3 report', 8000) for _ in range(25)] +
        [('Analyze strategic implications', 60000) for _ in range(15)]
    )
]

stats = batch_with_routing(sample_docs)
print('Routing distribution for 100 docs:')
for tier, info in stats.items():
    print(f'  {tier:>10}: {info["count"]:>3} docs -> {info["model"]}')

## Exercise 6: Per-tenant labels

**Difficulty:** Medium

Attach labels to every call, then write SQL to query per-tenant cost from the billing export.

1. Route the query, then call `generate_content` with `labels` (tenant, feature, tier, env).
2. Define a BigQuery template that `UNNEST`s the billing export labels and groups Vertex AI Gemini spend by tenant.

**Expected behaviour:** Labels propagate; the UNNEST query returns per-tenant monthly cost.

In [ ]:
# Cell 8 - Per-tenant cost attribution via request labels + BigQuery billing export
def tenant_aware_generate(tenant_id: str, query: str, feature: str = 'qa'):
    '''Generate a response tagged with tenant billing labels.'''
    model = rule_route(query, len(query))
    tier = model.split('-', 2)[-1]  # e.g. 'flash-lite'

    response = client.models.generate_content(
        model=model,
        contents=query,
        config=GenerateContentConfig(
            labels={
                'tenant':  tenant_id,
                'feature': feature,
                'tier':    tier,
                'env':     'production',
            }
        ),
    )
    return response

# Per-tenant cost query against the BigQuery billing export
QUERY_TENANT_COSTS = '''
SELECT
  labels.value AS tenant,
  ROUND(SUM(cost), 2) AS monthly_cost_usd,
  COUNT(*) AS request_count
FROM `{project}.{dataset}.gcp_billing_export_v1_{billing_account_id}`,
  UNNEST(labels) AS labels
WHERE labels.key = 'tenant'
  AND service.description = 'Vertex AI'
  AND sku.description LIKE '%Gemini%'
  AND invoice.month = @invoice_month
GROUP BY tenant
ORDER BY monthly_cost_usd DESC
'''

print('tenant_aware_generate ready - labels: tenant, feature, tier, env')
print('Per-tenant billing query template defined (run via BigQuery client)')

## Exercise 7: Cascade with quality gate

**Difficulty:** Challenge

Try Flash-Lite first, check the output for hedging phrases, escalate to Flash then Pro.

1. Define hedging phrases that signal the cheap model was not confident.
2. Call the cheapest model; if the answer hedges (or is empty), escalate up the cascade.
3. Stop at the first confident answer, or at Pro (top tier).

**Expected behaviour:** ~80% resolve at Lite, ~15% at Flash, ~5% escalate to Pro.

In [ ]:
# Exercise 7 - Cascade: cheapest model first, escalate on a quality gate
# (Lesson Step 5 pattern 3 - not a lesson-notebook cell; built on rule/router helpers above.)
HEDGE_PHRASES = [
    'i am not sure', "i'm not sure", 'it is unclear', 'cannot determine',
    'insufficient information', 'i cannot', 'unable to', 'not enough context',
    'more context', 'as an ai',
]
CASCADE = ['gemini-3.1-flash-lite', 'gemini-3.6-flash', 'gemini-3.1-pro-preview']

def looks_uncertain(text: str) -> bool:
    t = (text or '').lower().strip()
    if len(t) < 5:
        return True
    return any(p in t for p in HEDGE_PHRASES)

def cascade_generate(query: str, top_tier: int = 2) -> dict:
    '''Escalate up CASCADE until a confident answer or the top tier is reached.'''
    for i, model in enumerate(CASCADE[:top_tier + 1]):
        resp = client.models.generate_content(
            model=model,
            contents=query,
            config=GenerateContentConfig(temperature=0.2, max_output_tokens=400),
        )
        text = resp.text or ''
        if not looks_uncertain(text) or i == top_tier:
            return {'model': model, 'tier': i, 'escalations': i, 'answer': text}
    return {'model': CASCADE[top_tier], 'tier': top_tier, 'escalations': top_tier, 'answer': ''}

# Requires a real project/ADC to execute:
# result = cascade_generate('Extract the invoice total from: Total due Rs 4,200')
# print(f"resolved at {result['model']} after {result['escalations']} escalation(s)")
print('Cascade quality-gate function ready')
print('Expected steady-state: ~80% Lite, ~15% Flash, ~5% Pro')

## Exercise 8: Full cost dashboard

**Difficulty:** Challenge

Wire budget alerts (50 / 80 / 100 / 150%) to Pub/Sub to a Cloud Function that throttles a tenant when its cap is hit.

1. Create a Pub/Sub topic for budget notifications.
2. Create a billing budget with the four threshold rules publishing to that topic.
3. Deploy a Cloud Function subscribed to the topic that revokes the tenant's API key on a cap hit.

**Expected behaviour:** A cap hit triggers API key revocation via the Cloud Function.

In [ ]:
%%bash
# Exercise 8 (part 1) - Pub/Sub topic + billing budget with threshold alerts
# (Lesson Step 10 architecture - not a lesson-notebook cell; gcloud plumbing.)
PROJECT=documind-ai-YOUR-ID
BILLING_ACCOUNT=0X0X0X-0X0X0X-0X0X0X   # gcloud billing accounts list
TOPIC=budget-alerts

# 1. Topic that budget notifications publish to
gcloud pubsub topics create $TOPIC --project=$PROJECT

# 2. Budget with 50 / 80 / 100 / 150% thresholds -> Pub/Sub
gcloud billing budgets create \
  --billing-account=$BILLING_ACCOUNT \
  --display-name="DocuMind Gemini monthly cap" \
  --budget-amount=5000USD \
  --threshold-rule=percent=0.5 \
  --threshold-rule=percent=0.8 \
  --threshold-rule=percent=1.0 \
  --threshold-rule=percent=1.5 \
  --all-updates-rule-pubsub-topic=projects/$PROJECT/topics/$TOPIC

In [ ]:
%%bash
# Exercise 8 (part 2) - Cloud Function that throttles a tenant on a cap hit
# Writes a tiny handler, then deploys it subscribed to the budget topic.
PROJECT=documind-ai-YOUR-ID
TOPIC=budget-alerts
mkdir -p throttle_fn

cat > throttle_fn/main.py <<'PY'
import base64, json

def throttle_on_cap(event, context):
    """Triggered by a budget Pub/Sub message. Revoke the tenant key at 100%+."""
    payload = json.loads(base64.b64decode(event['data']).decode('utf-8'))
    spend = payload.get('costAmount', 0)
    budget = payload.get('budgetAmount', 1)
    ratio = spend / budget if budget else 0
    print(f'budget alert: {ratio:.0%} of cap (${spend:.2f} / ${budget:.2f})')
    if ratio >= 1.0:
        # Revoke / disable the tenant's API key so no further calls land.
        # e.g. gcloud api-keys update KEY --restrictions or Secret Manager rotation
        print('CAP HIT - revoking tenant API key')
PY

cat > throttle_fn/requirements.txt <<'REQ'
functions-framework
REQ

gcloud functions deploy throttle_on_cap \
  --project=$PROJECT \
  --runtime=python312 \
  --trigger-topic=$TOPIC \
  --source=throttle_fn \
  --entry-point=throttle_on_cap \
  --region=us-central1